# 03 Modeling and Hyperparameter Tuning

This notebook trains multiple regression models, compares split strategies, tunes selected models, and saves the best-performing artifacts.

## Modeling Goals

- train at least four regression models
- compare random split with chronological split
- tune at least two models
- identify the best model using RMSE and R2 Score

In [1]:
from pathlib import Path
import sys

import joblib
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import MODELS_DIR, PROCESSED_DIR, TABLES_DIR
from src.evaluate import format_results_table
from src.train import build_tuning_comparison, evaluate_tuned_models, run_model_suite

TABLES_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

## Load Modeling Dataset

This file is the practical training sample prepared during preprocessing.

In [2]:
data = pd.read_csv(PROCESSED_DIR / 'modeling_dataset.csv', parse_dates=['dt'])
print('Modeling dataset shape:', data.shape)
data.head()

Modeling dataset shape: (75000, 25)


,dt,AverageTemperature,AverageTemperatureUncertainty,City,Country,Latitude,Longitude,LatitudeValue,LongitudeValue,Year,...,YearsSince1900,RegionTag,Lag1Temperature,Lag12Temperature,Rolling12MeanTemperature,HistoricalCityMonthMean,TemperatureAnomaly,CityYearMeanTemperature,DecadeMeanTemperature,ClimateRiskIndex
0,1900-07-01,23.470,0.801,Cartagena,Spain,37.78N,0.00W,37.78,-0.00,1900,...,0,Global,29.283,12.283,21.741500,29.283,-5.813001,22.572792,17.346918,-3.813534
1,1900-08-01,26.240,0.364,Colombo,Sri Lanka,7.23N,80.27E,7.23,80.27,1900,...,0,South Asia,15.205,26.290,21.537250,15.205,11.035000,21.940708,17.346918,5.031665
2,1900-09-01,25.975,0.341,Colombo,Sri Lanka,7.23N,80.27E,7.23,80.27,1900,...,0,South Asia,16.276,27.888,21.148000,16.276,9.699001,21.940708,17.346918,4.117771
3,1900-10-01,11.560,0.479,Springfield,United States,42.59N,72.00W,42.59,-72.00,1900,...,0,Global,16.891,18.907,21.491667,16.729,-5.169000,11.294556,17.346918,-4.600680
4,1901-01-01,19.306,0.370,Taunggyi,Burma,20.09N,97.25E,20.09,97.25,1901,...,1,Global,19.812,19.009,23.673334,19.009,0.296999,23.684166,17.346918,-1.610227


## Baseline Models on Random Split

Random split is useful as a benchmark, but it may be optimistic for time-related climate data.

In [3]:
random_results, random_models = run_model_suite(data, split_strategy='random')
random_results

,split_strategy,model,tuned,mae,mse,rmse,r2
2,random,Random Forest,No,0.875363,1.720494,1.311676,0.982795
4,random,XGBoost,No,0.890146,1.743060,1.320250,0.982569
1,random,Decision Tree,No,0.958450,2.119022,1.455686,0.978809
3,random,Gradient Boosting,No,0.946740,2.153778,1.467575,0.978462
0,random,Linear Regression,No,0.995313,2.426533,1.557733,0.975734


Interpretation: Random split usually gives slightly better metrics because past and future patterns can mix more easily.

## Baseline Models on Chronological Split

Chronological split is more realistic because the model is trained on earlier data and tested on later data.

In [4]:
chrono_results, chrono_models = run_model_suite(data, split_strategy='chronological')
chrono_results

,split_strategy,model,tuned,mae,mse,rmse,r2
2,chronological,Random Forest,No,0.927169,1.882004,1.371861,0.980485
4,chronological,XGBoost,No,0.932164,1.894259,1.376321,0.980358
3,chronological,Gradient Boosting,No,0.964709,2.126050,1.458098,0.977954
0,chronological,Linear Regression,No,1.009243,2.390529,1.546133,0.975211
1,chronological,Decision Tree,No,1.040459,2.425386,1.557365,0.974850


Interpretation: These results are more suitable for the final report because they better reflect future-style prediction.

## Hyperparameter Tuning

The notebook tunes two stronger models: Random Forest and Gradient Boosting.

In [5]:
tuned_results, tuned_models, train_df, test_df, tuning_details = evaluate_tuned_models(data, split_strategy='chronological')
tuned_results

,split_strategy,model,tuned,mae,mse,rmse,r2
1,chronological,Gradient Boosting Tuned,Yes,0.959657,2.126478,1.458245,0.977950
0,chronological,Random Forest Tuned,Yes,0.981599,2.230528,1.493495,0.976871


Interpretation: Tuning improves the rigor of the workflow, even if the untuned Random Forest remains the strongest final baseline.

## Combined Comparison Table

This is the main result table used in the report.

In [6]:
combined_results = pd.concat([random_results, chrono_results, tuned_results], ignore_index=True)
comparison_table = format_results_table(combined_results)
comparison_table.to_csv(TABLES_DIR / 'model_comparison_results.csv', index=False)
comparison_table

,Split Strategy,Model,Tuned,MAE,MSE,RMSE,R2 Score
0,random,Random Forest,No,0.875,1.720,1.312,0.983
1,random,XGBoost,No,0.890,1.743,1.320,0.983
2,random,Decision Tree,No,0.958,2.119,1.456,0.979
3,random,Gradient Boosting,No,0.947,2.154,1.468,0.978
4,random,Linear Regression,No,0.995,2.427,1.558,0.976
5,chronological,Random Forest,No,0.927,1.882,1.372,0.980
6,chronological,XGBoost,No,0.932,1.894,1.376,0.980
7,chronological,Gradient Boosting,No,0.965,2.126,1.458,0.978
8,chronological,Linear Regression,No,1.009,2.391,1.546,0.975
9,chronological,Decision Tree,No,1.040,2.425,1.557,0.975


## Best Hyperparameters and Before/After Tuning

These tables make the tuning stage clearer for the report.

In [7]:
tuning_details.to_csv(TABLES_DIR / 'best_hyperparameters.csv', index=False)
tuning_details

,Model,Search Method,CV Folds,Iterations,Best CV RMSE,Best Parameters
0,Gradient Boosting Tuned,RandomizedSearchCV,3,4,1.4225,"{""model__learning_rate"": 0.08, ""model__max_dep..."
1,Random Forest Tuned,RandomizedSearchCV,3,4,1.4889,"{""model__max_depth"": 12, ""model__max_features""..."


In [8]:
chrono_baseline_subset = chrono_results[chrono_results['model'].isin(['Random Forest', 'Gradient Boosting'])].copy()
tuning_comparison = build_tuning_comparison(chrono_baseline_subset, tuned_results)
tuning_comparison.to_csv(TABLES_DIR / 'tuning_before_after_comparison.csv', index=False)
tuning_comparison

,Baseline Model,Tuned Model,Baseline RMSE,Tuned RMSE,RMSE Change,Baseline R2,Tuned R2,R2 Change
0,Random Forest,Random Forest Tuned,1.3719,1.4935,0.1216,0.9805,0.9769,-0.0036
1,Gradient Boosting,Gradient Boosting Tuned,1.4581,1.4582,0.0001,0.9780,0.9779,-0.0000


## Best Model Selection

The best chronological baseline and best tuned model are saved for later evaluation.

In [9]:
best_baseline_name = chrono_results.sort_values('rmse').iloc[0]['model']
best_tuned_name = tuned_results.sort_values('rmse').iloc[0]['model']

joblib.dump(chrono_models[best_baseline_name], MODELS_DIR / 'best_baseline_model.joblib')
joblib.dump(tuned_models[best_tuned_name], MODELS_DIR / 'best_tuned_model.joblib')
train_df.to_csv(PROCESSED_DIR / 'chronological_train_split.csv', index=False)
test_df.to_csv(PROCESSED_DIR / 'chronological_test_split.csv', index=False)

print('Best baseline model:', best_baseline_name)
print('Best tuned model:', best_tuned_name)

Best baseline model: Random Forest
Best tuned model: Gradient Boosting Tuned


Final note: For this project, Random Forest under chronological evaluation is the strongest main result.